# Introduction

This notebook aims to understand and visualize static GTFS data reported from the TTC
Specifically we would like to visualize Route 29, a bus route that is known to have significantly high trip times

You can find the exact data source used in this project below: https://open.toronto.ca/dataset/merged-gtfs-ttc-routes-and-schedules/



# Data Modeling

Before we dig into the data itself, we will start off by understanding the inherent structure of the data (the data model). The GTFS specifciation is a global standard used to structure transit data reported by transit agencies. Specifically the GTFS specifies an array of CSV text files each representing different types of transit information (i.e. route/trip information, schedule information, geo-shape information etc.). Each CSV text file relate to each other through a relational schema.

A relational schema representing the GTFS data reported by the TTC can be found below:


![GTFS_STATIC_Relational_Model](/Users/devrajsolanki/Documents/TAL/GTFS_STATIC_Relational_Model.jpeg)

# Data Preprocessing

GTFS feeds can be quite messy at times, containing incomplete, invalid and inconsistent data.
Thankfully there is significant effort to validate the quality of GTFS feed data.
 
We will be using validation techniques suggested from Mobility Data (https://gtfs-validator.mobilitydata.org/rules.html) as well as common 
Data Preprocessing/Cleaning techniques.



### Data Loading

In [795]:
import pandas as pd

In [796]:
stops = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stops.txt")
routes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/routes.txt")
trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")
shapes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/shapes.txt")
calendar = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar.txt")
agency = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/agency.txt")
calendar_dates = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar_dates.txt")
route_types = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/route_types.txt")
feed_info = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/feed_info.txt")

/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_73779/2009019488.py:3: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_73779/2009019488.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")


### Data Understanding and Cleaning 

The inclusion of a stop_times.txt file means this is a schedule-based feed (define specific arrival and departure times for every trip at every stop)

In [797]:
# lets look at the shape of each dataframe to understand the size of the data

print("Shape of stops dataframe:", stops.shape)
print("Shape of routes dataframe:", routes.shape)
print("Shape of trips dataframe:", trips.shape)
print("Shape of stop_times dataframe:", stop_times.shape)
print("Shape of shapes dataframe:", shapes.shape)
print("Shape of calendar dataframe:", calendar.shape)
print("Shape of agency dataframe:", agency.shape)
print("Shape of calendar_dates dataframe:", calendar_dates.shape)
print("Shape of route_types dataframe:", route_types.shape)
print("Shape of feed_info dataframe:", feed_info.shape)

# It seems that the number of records for each file matches logic; there should be a lot of stop_times records since each trip has multiple 
# stops and there should be fewer routes than trips, etc. This is a good sign that the feed is complete.

Shape of stops dataframe: (9423, 13)
Shape of routes dataframe: (230, 9)
Shape of trips dataframe: (135412, 9)
Shape of stop_times dataframe: (4283994, 10)
Shape of shapes dataframe: (1020671, 5)
Shape of calendar dataframe: (8, 10)
Shape of agency dataframe: (1, 9)
Shape of calendar_dates dataframe: (9, 3)
Shape of route_types dataframe: (679, 5)
Shape of feed_info dataframe: (1, 6)


#### Stops.txt

In [798]:
# lets look at the data types of each column in each dataframe to identify if the feed is clean/recorded properly

print("Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_desc              float64
stop_lat               float64
stop_lon               float64
zone_id                float64
stop_url               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [799]:
# its weird that stop_desc is a float and not a string/object, so lets look at the unique values in the column
print("Unique values in stop_desc column of stops dataframe:")
print(stops["stop_desc"].unique())

# so it seems that the TTC Feed leaves out/excludes stop description, so lets remove the column from the dataframe
stops = stops.drop(columns=["stop_desc"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Unique values in stop_desc column of stops dataframe:
[nan]
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
zone_id                float64
stop_url               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [800]:
# everything checks out, lets now look at the distribution of values for each column as well
# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

# stop_id
print("Distribution of stop_id column in stops dataframe:")
print(stops["stop_id"].value_counts())
print("-" * 50)

# There are 9423 unique stop_ids which matches the number of records in the stop dataframe so there are no duplicate primary keys

# stop_code
print("Distribution of stop_code column in stops dataframe:")
print(stops["stop_code"].value_counts())
print("-" * 50)

# There also seems to be 9423 unique stop_codes per record

Distribution of stop_id column in stops dataframe:
stop_id
1        1
8449     1
8430     1
8431     1
8432     1
        ..
4742     1
4743     1
4745     1
4746     1
16803    1
Name: count, Length: 9423, dtype: int64
--------------------------------------------------
Distribution of stop_code column in stops dataframe:
stop_code
7978     1
5831     1
12194    1
990      1
2395     1
        ..
3959     1
8896     1
1876     1
7166     1
16803    1
Name: count, Length: 9423, dtype: int64
--------------------------------------------------


In [801]:
# stop_name
print("Distribution of stop_name column in stops dataframe:")
print(stops["stop_name"].value_counts())
print("-" * 50)

# filter stop_name for "Centennial College Bus Terminal"
centennial_stops = stops[stops["stop_name"] == "Centennial College Bus Terminal"]
print("Filtered stops for 'Centennial College Bus Terminal':")
print(centennial_stops)

# So it seems that some stops share the same name but have different stop_ids and stop_codes and even slightly different coordinates. 
# This makes sense since there can be multiple platforms at the same general location (Centennial college has 4 bus platforms).

Distribution of stop_name column in stops dataframe:
stop_name
Centennial College Bus Terminal                                 4
Humber College Bus Terminal                                     4
The Queensway at South Kingsway                                 4
Bathurst St at King St West                                     4
Bathurst St at Niagara St                                       4
                                                               ..
Millwood Rd at Southvale Dr                                     1
Belfield Rd at City View Dr                                     1
Faywood Blvd at Invermay Ave                                    1
Queens Quay West at Bathurst St East Side Billy Bishop Airpo    1
McCowan Rd at Highway 401                                       1
Name: count, Length: 7782, dtype: int64
--------------------------------------------------
Filtered stops for 'Centennial College Bus Terminal':
      stop_id  stop_code                        stop_name   stop_l

In [802]:
# stop_lat
print("Distribution of stop_lat column in stops dataframe:")
print("Range of stop_lat:", stops["stop_lat"].min(), "to", stops["stop_lat"].max())
print("-" * 50)

# stop_lon
print("Distribution of stop_lon column in stops dataframe:")
print("Range of stop_lon:", stops["stop_lon"].min(), "to", stops["stop_lon"].max())
print("-" * 50)

# Toronto lies around 43.6425, -79.387222 [wikipedia] so the stop coordinates seem to be within the expected range for Toronto.

Distribution of stop_lat column in stops dataframe:
Range of stop_lat: 43.59181 to 43.90975
--------------------------------------------------
Distribution of stop_lon column in stops dataframe:
Range of stop_lon: -79.649908 to -79.123111
--------------------------------------------------


In [803]:
# zone_id
print("Distribution of zone_id column in stops dataframe:")
print(stops["zone_id"].value_counts())
print("-" * 50)

# It seems that the zone_id column is not used in the feed since all values are NaN,

# stop_url
print("Distribution of stop_url column in stops dataframe:")
print(stops["stop_url"].value_counts())
print("-" * 50)

# It seems that the stop_url column is not used in the feed since all values are NaN

# remove the zone_id and stop_url columns since they are not used in the feed
stops = stops.drop(columns=["zone_id", "stop_url"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Distribution of zone_id column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Distribution of stop_url column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [804]:
# location_type
print("Distribution of location_type column in stops dataframe:")
print(stops["location_type"].value_counts())
print("-" * 50)

# so it seems that the TTC reported all stops as location_type 0 (stops/platforms); this means that stations, entrances/exits are 
# not included in the feed data; just the points where the bus/subway etc. picks up/drops off passengers

# parent_station
print("Distribution of parent_station column in stops dataframe:")
print(stops["parent_station"].value_counts())
print("-" * 50)

# hence parent_station has all values are NaN since stations are not included in the feed data 

# drop parent_station column since it is not used in the feed
stops = stops.drop(columns=["parent_station"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)

Distribution of location_type column in stops dataframe:
location_type
0    9423
Name: count, dtype: int64
--------------------------------------------------
Distribution of parent_station column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [805]:
# stop_timezone
print("Range of stop_timezone column in stops dataframe:")
print(stops["stop_timezone"].min(), stops["stop_timezone"].max())

# it seems that the stop_timezone column is not used in the feed since all values are NaN, so we will drop the column from the dataframe
stops = stops.drop(columns=["stop_timezone"])

# wheelchair_boarding
print("Distribution of wheelchair_boarding column in stops dataframe:")
print(stops["wheelchair_boarding"].value_counts())
print("-" * 50)

# so it looks like most stops support wheelchair boarding (value 1), but some do not (value 2), and there are a few that have unknown wheelchair boarding status (value 0) 

# level_id
print("Distribution of level_id column in stops dataframe:")
print(stops["level_id"].value_counts())
print("-" * 50)

# it seems that the level_id column is not used in the feed since all values are NaN, so we will drop the column from the dataframe
stops = stops.drop(columns=["level_id"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)

Range of stop_timezone column in stops dataframe:
nan nan
Distribution of wheelchair_boarding column in stops dataframe:
wheelchair_boarding
1    8020
2    1396
0       7
Name: count, dtype: int64
--------------------------------------------------
Distribution of level_id column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
wheelchair_boarding      int64
dtype: object
--------------------------------------------------


In [806]:
# lets look at the presence of null values and empty strings in the rows of the stop 

print("Number of null values in each column of stops dataframe:")
print(stops.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of stops dataframe:")
object_columns = stops.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (stops[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# overall, looks like the feed for stops has complete data

Number of null values in each column of stops dataframe:
stop_id                0
stop_code              0
stop_name              0
stop_lat               0
stop_lon               0
location_type          0
wheelchair_boarding    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of stops dataframe:
stop_name: 0
--------------------------------------------------


#### Routes.txt

We will now perform the same cleaning process to the routes table

In [807]:
# print data types of routes dataframe

print("Data types of routes dataframe:")
print(routes.dtypes)
print("-" * 50)

Data types of routes dataframe:
route_id              int64
agency_id             int64
route_short_name      int64
route_long_name      object
route_desc          float64
route_type            int64
route_url           float64
route_color          object
route_text_color     object
dtype: object
--------------------------------------------------


In [808]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in routes.columns:
    unique_values = routes[col].unique()
    print(f"Unique values in {col} column of routes dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in route_id column of routes dataframe:
[ 10 100 101 102 103 104 105 106 107 108 109  11 110 111 112 113 114 115
 116 117 118 119  12 120 121 122 123 124 125 126 127 129  13 130 131 132
 133 134 135  14 149  15 151 154 156 158  16 160 161 162 164 165 166 167
 168 169  17 171  18 184 185 189  19 191  20 203  21  22  23  24  25  26
  27  28  29  30 300 301 304 305 306 307  31 310 312 315 316  32 320 322
 324 325 329  33 334 335 336 337 339  34 340 341 343  35 352 353 354  36
 363 365  37  38 384 385 386  39 395 396  40  41  42  43  44  45  46  47
  48  49  50 501 503 504 505 506 507 508 509  51 510 511 512  52  53  54
  55  57  59  60  61  62  63  64  65  66  67  68  69   7  70  71  72  73
  74  75  76  77  78  79   8  80 805  82  83 830  84  85  86  87  88  89
   9  90 900 902 903 904 905 906  91  92 924 925 927 929  93 935 937 938
 939  94 941 944 945  95 952 953 954  96 960 968  97  98 984 985 986 989
  99 995 996   1   2   4 400 402 403 404 405 406   5   6]
------------

In [809]:
# drop route_desc and route_url columns since they are not used in the feed
routes = routes.drop(columns=["route_desc", "route_url"])

print("Updated Data types of routes dataframe:")
print(routes.dtypes)
print("-" * 50)

Updated Data types of routes dataframe:
route_id             int64
agency_id            int64
route_short_name     int64
route_long_name     object
route_type           int64
route_color         object
route_text_color    object
dtype: object
--------------------------------------------------


In [810]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

for col in routes.columns:
    if col == "route_id":
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)
    elif routes[col].dtype == "int64":
        print(f"Range of {col} column in routes dataframe: {routes[col].min()} to {routes[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)
    elif routes[col].dtype == "float64":
        print(f"Range of {col} column in routes dataframe: {routes[col].min()} to {routes[col].max()}")
        print("-" * 50)
    elif routes[col].dtype == "object":
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)

Distribution of route_id column in routes dataframe:
route_id
10     1
55     1
59     1
60     1
61     1
      ..
306    1
307    1
31     1
310    1
6      1
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Range of agency_id column in routes dataframe: 1 to 1
--------------------------------------------------
Distribution of agency_id column in routes dataframe:
agency_id
1    230
Name: count, dtype: int64
--------------------------------------------------
Range of route_short_name column in routes dataframe: 1 to 996
--------------------------------------------------
Distribution of route_short_name column in routes dataframe:
route_short_name
10     1
55     1
59     1
60     1
61     1
      ..
306    1
307    1
31     1
310    1
6      1
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Distribution of route_long_name column in routes dataframe:
route_long_name
Bathurst            3
Lawrence East  

In [811]:
# There are some obervations we can make from this

# 1) There are 230 unique rows in the routes dataframe; each row seems to correspnd to a unique value of route_id (230 unique route_ids). This is a good sign
#    that the primary key is unique

# 2) agency_id contains only 1 unique value (1) meaning that the only transit agency managing routes in this feed is the TTC

# 3) it seems each route has a unique route_short_name (equal in value to route_id) but some routes share the same route_long_name
#    This makes sense since the same bus routes, for example, can have different route numbers but end up at the same destination
#.   For example route 307 and 511 are different routes but both end up at Bathurst station, so they share the same route_long_name

# 4) There are 3 unique route_types in the TTC feed; 3 (bus), 1 (subway), and 4 (streetcar) which makes sense since the TTC operates these 3 modes of transportation
#    There are more bus routes than subway and streetcar routes which also makes sense since the bus network is usually more extensive than subway and streetcar networks in a city
#.   The 3 subway routes line up with the 3 subway lines in Toronto (Yonge-University-Spadina, Bloor-Danforth, and Sheppard) which is another good sign that the feed is accurate and complete


In [812]:
# looking at routes with route_long_name = "Bathurst"
print("Routes with route_long_name 'Bathurst':")
print(routes[routes["route_long_name"] == "Bathurst"])
print("-" * 50)

# looking at routes with route_type = 1 (subway)
print("Routes with route_type 1 (subway):")
print(routes[routes["route_type"] == 1])
print("-" * 50)


Routes with route_long_name 'Bathurst':
     route_id  agency_id  route_short_name route_long_name  route_type  \
81        307          1               307        Bathurst           3   
139       511          1               511        Bathurst           0   
157         7          1                 7        Bathurst           3   

    route_color route_text_color  
81       0054A6           FFFFFF  
139      ED1C24           FFFFFF  
157      ED1C24           FFFFFF  
--------------------------------------------------
Routes with route_type 1 (subway):
     route_id  agency_id  route_short_name        route_long_name  route_type  \
219         1          1                 1  Yonge-University Line           1   
220         2          1                 2    Bloor-Danforth Line           1   
221         4          1                 4          Sheppard Line           1   

    route_color route_text_color  
219      D5C82B           000000  
220      008000           FFFFFF  
221    

In [813]:
# lets look at the presence of null values and empty strings in the rows of the routes dataframe

print("Number of null values in each column of routes dataframe:")
print(routes.isnull().sum())
print("-" * 50)

# lets look at the presence of null values and empty strings in the rows of the routes dataframe
print("Number of empty strings in object dtype columns of routes dataframe:")
object_columns = routes.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (routes[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# overall, looks like the feed for routes has complete data

Number of null values in each column of routes dataframe:
route_id            0
agency_id           0
route_short_name    0
route_long_name     0
route_type          0
route_color         0
route_text_color    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of routes dataframe:
route_long_name: 0
route_color: 0
route_text_color: 0
--------------------------------------------------


#### Trips.txt

In [814]:
# print data types of trips dataframe

print("Data types of trips dataframe:")
print(trips.dtypes)
print("-" * 50)

Data types of trips dataframe:
trip_id                   int64
route_id                  int64
service_id                int64
trip_headsign            object
trip_short_name          object
direction_id              int64
block_id                  int64
shape_id                 object
wheelchair_accessible     int64
dtype: object
--------------------------------------------------


In [815]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in trips.columns:
    unique_values = trips[col].unique()
    print(f"Unique values in {col} column of trips dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in trip_id column of trips dataframe:
[106986020 118048020 102340020 ... 133251418 133251419 133251420]
--------------------------------------------------
Unique values in route_id column of trips dataframe:
[ 10 100 101 102 103 104 105 106 107 108 109  11 110 111 112 113 114 115
 116 117 118 119  12 120 121 122 123 124 125 126 127 129  13 130 131 132
 133 134 135  14 149  15 151 154 156 158  16 160 161 162 164 165 166 167
 168 169  17 171  18 184 185 189  19 191  20 203  21  22  23  24  25  26
  27  28  29  30 300 301 304 305 306 307  31 310 312 315 316  32 320 322
 324 325 329  33 334 335 336 337 339  34 340 341 343  35 352 353 354  36
 363 365  37  38 384 385 386  39 395 396  40  41  42  43  44  45  46  47
  48  49  50 501 503 504 505 506 507 508 509  51 510 511 512  52  53  54
  55  57  59  60  61  62  63  64  65  66  67  68  69   7  70  71  72  73
  74  75  76  77  78  79   8  80 805  82  83 830  84  85  86  87  88  89
   9  90 900 902 903 904 905 906  91  92 924 925

In [816]:
# its weird that the shape_id column is an object column containing both string values and integer values
# lets look at the unique values in the shape_id column to understand why this is the case
print("Unique values in shape_id column of trips dataframe:")
print(trips["shape_id"].unique().tolist())

# it looks like shape_id either takes the format: string "shp-###-##" or integer ##### or string integer
# "#####"

# lets see if the same shape_id is represented as both an integer and a string in different records of trip

display(trips[(trips["shape_id"] == 1103634)].head(3))
display(trips[(trips["shape_id"] == "1103634")].head(3))

# lets standardize the shape_id column so that all values are strings to avoid redundancy in the trips table

trips["shape_id"] = trips["shape_id"].astype(str)



Unique values in shape_id column of trips dataframe:
['shp-10-03', 'shp-10-04', 'shp-10-51', 'shp-10-01', 'shp-10-02', 'shp-10-52', 'shp-100-02', 'shp-100-01', 'shp-100-04', 'shp-100-03', 'shp-100-52', 'shp-100-54', 'shp-100-51', 'shp-100-53', 'shp-101-02', 'shp-101-03', 'shp-101-07', 'shp-101-01', 'shp-101-06', 'shp-101-05', 'shp-101-57', 'shp-101-56', 'shp-101-54', 'shp-101-52', 'shp-101-58', 'shp-101-53', 'shp-101-51', 'shp-102-05', 'shp-102-01', 'shp-102-06', 'shp-102-03', 'shp-102-02', 'shp-102-08', 'shp-102-11', 'shp-102-56', 'shp-102-57', 'shp-102-59', 'shp-102-54', 'shp-102-51', 'shp-102-52', 'shp-102-55', 'shp-103-01', 'shp-103-52', 'shp-103-51', 'shp-104-02', 'shp-104-52', 'shp-105-07', 'shp-105-55', 'shp-106-03', 'shp-106-01', 'shp-106-54', 'shp-106-57', 'shp-107-33', 'shp-107-07', 'shp-107-73', 'shp-107-53', 'shp-108-05', 'shp-108-52', 'shp-108-06', 'shp-108-54', 'shp-108-53', 'shp-108-55', 'shp-108-56', 'shp-108-04', 'shp-109-08', 'shp-109-06', 'shp-109-09', 'shp-109-07', 

,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
131072,133247081,4,2,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203384,1103634,1
131073,133247082,4,2,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203385,1103634,1
131074,133247083,4,2,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203386,1103634,1


,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
128813,133244822,4,1,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203380,1103634,1
128814,133244823,4,1,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203379,1103634,1
128815,133244824,4,1,Sheppard Line towards Sheppard-Yonge Station,NaN,1,2203382,1103634,1


In [817]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

for col in trips.columns:
    if col == "trip_id":
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)
    elif trips[col].dtype == "int64":
        print(f"Range of {col} column in trips dataframe: {trips[col].min()} to {trips[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)
    elif trips[col].dtype == "float64":
        print(f"Range of {col} column in trips dataframe: {trips[col].min()} to {trips[col].max()}")
        print("-" * 50)
    elif trips[col].dtype == "object":
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)


Distribution of trip_id column in trips dataframe:
trip_id
106986020    1
80251020     1
11708020     1
126289070    1
117554020    1
            ..
18410080     1
86115070     1
39406010     1
10436080     1
133251420    1
Name: count, Length: 135412, dtype: int64
--------------------------------------------------
Range of route_id column in trips dataframe: 1 to 996
--------------------------------------------------
Distribution of route_id column in trips dataframe:
route_id
504    3246
63     2533
47     2393
1      2276
2      2206
       ... 
406      16
405      14
400      14
403      12
830       2
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Range of service_id column in trips dataframe: 1 to 7001
--------------------------------------------------
Distribution of service_id column in trips dataframe:
service_id
1       39810
2       33519
3       30298
5       29810
4        1929
4501       16
7001       16
4401       14
Name: coun

In [818]:
# There are some obervations we can make from this

# 1) There are 135412 unique rows in the trips dataframe; each row seems to correspnd to a unique value of trip_id (135412 unique trip_ids)
#    This is a good sign that the primary key is unique

# 2) The route_id column contains 230 unique values ranging from 1 to 996. This matches the number of unique route_ids in the routes
#    dataframe and the range of route_id's in the routes dataframe. This means all routes will have one or more trips scheduled, which is a good
#    sign that the feed is complete

# 3) it seems that all scheduled trips will follow 8 possible weekly schedules (service_id's). 

# 4) it seems many trips share the same trip_headsign (ranging from 1 to thousands sharing the same headsign), this makes sense since
#    many trips from different routes, scheduled at different times, can share the same destination and therefore the same headsign

# 5) Most trips do not have a specified trip_short_name (value NaN) but the rest seem to have an arbritary letter assigned as a trip_short_name
#    There doesnt seem to be much documentation of what exactly the trip_short_name represents in the TTC feed, but they seem to refer to the
#.   letter associated with a bus trip in the TTC system i.e. the 102 "D" (see below)

# 6) The range of direction_id is from 0 to 1 which makes sense since direction_id entails the direction of travel of the trip 
#    (0 vs 1 meaning eastbound vs westbound, etc.)

# 7) block_id represents the block of sequential trips made by the same vehicle, so it makes sense that there are fewer unique blocks than trips.
#    it is suprising that hundreds of trips share the same block_id, but this could just be looped routes or the same sequence of routes being 
#    travelled through by different veichles at different schedules

# 8) its weird that there are different id formats in the shape_id column, but as long as the ids are consistent and unique and correspond to the
#    same id's in the shapes dataframe, then it shouldnt be much of an issue for analysis

# 9) it seems that the TTC completely specifies wheel chair accesibility for each trip (value 1 or 2)

In [819]:
# what is the range of route_id values in the routes dataframe
print("Range of route_id values in routes dataframe:")
print(routes["route_id"].min(), "to", routes["route_id"].max())
print("-" * 50)

# print out records with a trip_short_name of "D" 
print("Trips with trip_short_name 'D':")
print(trips[trips["trip_short_name"] == "D"].head(10).to_string())
print("-" * 50)

Range of route_id values in routes dataframe:
1 to 996
--------------------------------------------------
Trips with trip_short_name 'D':
       trip_id  route_id  service_id                                    trip_headsign trip_short_name  direction_id  block_id    shape_id  wheelchair_accessible
2005  74998070       102           2  North - 102D Markham Rd towards Major Mackenzie               D             1  10222220  shp-102-08                      1
2006  23309020       102           1  North - 102D Markham Rd towards Major Mackenzie               D             1   1020990  shp-102-08                      1
2007  71179080       102           5  North - 102D Markham Rd towards Major Mackenzie               D             1  10214140  shp-102-08                      1
2008  23577070       102           2  North - 102D Markham Rd towards Major Mackenzie               D             1  10214140  shp-102-08                      1
2009  75263020       102           1  North - 102D Markha

In [820]:
# lets look at the presence of null values and empty strings in the rows of the trips dataframe

print("Number of null values in each column of trips dataframe:")
print(trips.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of trips dataframe:")
object_columns = trips.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (trips[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

Number of null values in each column of trips dataframe:
trip_id                      0
route_id                     0
service_id                   0
trip_headsign                0
trip_short_name          92831
direction_id                 0
block_id                     0
shape_id                     0
wheelchair_accessible        0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of trips dataframe:
trip_headsign: 0
trip_short_name: 0
shape_id: 0
--------------------------------------------------


In [821]:
# so it seems like there are missing values in the trip_short_name column

# lets reprint the count of unique values in the trip_short_name column
print("Distribution of trip_short_name column in trips dataframe:")
print(trips["trip_short_name"].value_counts(dropna=False))
print("-" * 50)

# This is most likely due to some veichles not needing a letter designation for the trip name (i.e. it may be bus 100 not bus 100 "A")

# print out some records with NaN trip_short_name values and some records with A trip_short_name values to compare
print("Trips with NaN trip_short_name:")
display(trips[trips["trip_short_name"].isna()].head(3))

print("Trips with trip_short_name 'A':")
display(trips[trips["trip_short_name"] == "A"].head(3))

# so it seems that the initial hypothesis was true, instead of leaving the trip_short_name blank for trips that dont have a letter designation.
# lets fill the NaN values in the trip_short_name column with "No Designation" to make it more clear that these trips do not have a short name designation instead of leaving it as blank/NaN

trips["trip_short_name"] = trips["trip_short_name"].fillna("No Designation")

print("-" * 50)
print("Updated distribution of trip_short_name column in trips dataframe:")
print(trips["trip_short_name"].value_counts(dropna=False))
print("-" * 50)

Distribution of trip_short_name column in trips dataframe:
trip_short_name
NaN    92831
A      21568
B      11275
C       4714
D       2915
S       1554
G        439
F        114
E          2
Name: count, dtype: int64
--------------------------------------------------
Trips with NaN trip_short_name:


,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
0,106986020,10,1,East - 10 Van Horne towards Victoria Park,NaN,0,100112,shp-10-03,1
1,118048020,10,1,East - 10 Van Horne towards Victoria Park,NaN,0,100111,shp-10-03,1
2,102340020,10,1,East - 10 Van Horne towards Victoria Park,NaN,0,100112,shp-10-03,1


Trips with trip_short_name 'A':


,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible
42,1093010,100,3,North - 100A Flemingdon Park towards Don Valle...,A,1,1000880,shp-100-02,1
43,111645010,100,3,North - 100A Flemingdon Park towards Don Valle...,A,1,1000550,shp-100-02,1
44,109437070,100,2,North - 100A Flemingdon Park towards Don Valle...,A,1,1000440,shp-100-02,1


--------------------------------------------------
Updated distribution of trip_short_name column in trips dataframe:
trip_short_name
No Designation    92831
A                 21568
B                 11275
C                  4714
D                  2915
S                  1554
G                   439
F                   114
E                     2
Name: count, dtype: int64
--------------------------------------------------


In [822]:
# lets now re-look at the presence of null values and empty strings in the rows of the trips dataframe

print("Number of null values in each column of trips dataframe:")
print(trips.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of trips dataframe:")
object_columns = trips.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (trips[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# overall, looks like the feed for stops has complete data

Number of null values in each column of trips dataframe:
trip_id                  0
route_id                 0
service_id               0
trip_headsign            0
trip_short_name          0
direction_id             0
block_id                 0
shape_id                 0
wheelchair_accessible    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of trips dataframe:
trip_headsign: 0
trip_short_name: 0
shape_id: 0
--------------------------------------------------


#### Stop_times.txt

In [823]:
# print data types of stop_times dataframe

print("Data types of stop_times dataframe:")
print(stop_times.dtypes)
print("-" * 50)

Data types of stop_times dataframe:
trip_id                  int64
arrival_time            object
departure_time          object
stop_id                  int64
stop_sequence            int64
stop_headsign           object
pickup_type              int64
drop_off_type            int64
shape_dist_traveled    float64
timepoint              float64
dtype: object
--------------------------------------------------


In [824]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in stop_times.columns:
    unique_values = stop_times[col].unique()
    print(f"Unique values in {col} column of stop_times dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in trip_id column of stop_times dataframe:
[ 10000080 100001020 100002020 ... 133250515 133250516 133250424]
--------------------------------------------------
Unique values in arrival_time column of stop_times dataframe:
['20:25:53' '20:27:14' '20:29:21' ... '04:40:02' '28:47:21' '28:55:41']
--------------------------------------------------
Unique values in departure_time column of stop_times dataframe:
['20:25:53' '20:27:14' '20:29:21' ... '04:40:02' '28:47:21' '28:55:41']
--------------------------------------------------
Unique values in stop_id column of stop_times dataframe:
[ 4386 38709 42823 ... 13750 13751 13754]
--------------------------------------------------
Unique values in stop_sequence column of stop_times dataframe:
[  2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19
   1  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  

In [825]:
# its weird that arrival_time and departure_time are object dtypes instead of a datetime dtype

# lets look at the distribution of values in the arrival_time and departure_time columns
print("Distribution of arrival_time column in stop_times dataframe:")
print(stop_times["arrival_time"].value_counts(dropna=False))
print("-" * 50)
print("Distribution of departure_time column in stop_times dataframe:")
print(stop_times["departure_time"].value_counts(dropna=False))
print("-" * 50)

# lets look at the range of values in the arrival_time and departure_time columns 
print("Range of arrival_time column in stop_times dataframe:")
print(stop_times["arrival_time"].min(), "to", stop_times["arrival_time"].max())
print("-" * 50)
print("Range of departure_time column in stop_times dataframe:")
print(stop_times["departure_time"].min(), "to", stop_times["departure_time"].max())
print("-" * 50)

# so it seems that the arrival_time and departure_time columns record time since the start of service day, in a HH:MM:SS format
# this time can exceed the 24 hour mark since some trips can run past midnight so some arrival and departure times can be greater than 
# 24:00:00 (for example 25:30:00 would mean 1:30am the next day)

# lets see if there is a difference between the recorded arrival and depature times for the same stop_time record
display(stop_times[stop_times["arrival_time"] != stop_times["departure_time"]])

# so it seems that schedule built by the TTC assumes instantaneous arrival and departure at stops since all records have the same arrival
# and departure times

# I will keep the HH:MM:SS format for the arrival_time and departure_time columns since it is a standard format for recording time in GTFS feeds,
# and it can be easily converted to a datetime format if needed for analysis

Distribution of arrival_time column in stop_times dataframe:
arrival_time
25:00:00    461
12:00:00    425
19:10:00    413
19:00:00    411
10:00:00    409
           ... 
28:29:44      1
29:11:49      1
29:10:06      1
29:06:30      1
28:55:41      1
Name: count, Length: 92155, dtype: int64
--------------------------------------------------
Distribution of departure_time column in stop_times dataframe:
departure_time
25:00:00    461
12:00:00    425
19:10:00    413
19:00:00    411
10:00:00    409
           ... 
28:29:44      1
29:11:49      1
29:10:06      1
29:06:30      1
28:55:41      1
Name: count, Length: 92155, dtype: int64
--------------------------------------------------
Range of arrival_time column in stop_times dataframe:
03:28:00 to 30:46:41
--------------------------------------------------
Range of departure_time column in stop_times dataframe:
03:28:00 to 30:46:41
--------------------------------------------------


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint


In [826]:
# it is also weird that timepoint is a float dtype when it should be an int dtype (0 vs 1)

# lets display the frequencies of unique values in the timepoint column to see if it only takes the values 0 and 1 as expected
print("Distribution of timepoint column in stop_times dataframe:")
print(stop_times["timepoint"].value_counts(dropna=False))
print("-" * 50)

# so there seems to be some records with empty/null timepoint values. According to GTFS documentation, "All records of stop_times.txt with 
# defined arrival or departure times should have timepoint values populated. If no timepoint values are provided, all times are considered exact."
# since the TTC feed provides defined arrival and departure times for all stop_time records, we will impute the null values to 1 
# meaning that all stop times are considered exact time points in the schedule

# source: https://gtfs.org/documentation/schedule/reference/#routestxt

stop_times["timepoint"] = stop_times["timepoint"].fillna(1)

# we will also convert the timepoint column to an integer dtype since it should only take the values 0 and 1
stop_times["timepoint"] = stop_times["timepoint"].astype(int)

print("Updated distribution of timepoint column in stop_times dataframe:")
print(stop_times["timepoint"].value_counts(dropna=False))
print("-" * 50)

Distribution of timepoint column in stop_times dataframe:
timepoint
0.0    3801988
1.0     276242
NaN     205764
Name: count, dtype: int64
--------------------------------------------------
Updated distribution of timepoint column in stop_times dataframe:
timepoint
0    3801988
1     482006
Name: count, dtype: int64
--------------------------------------------------


In [827]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

# groupby trip_id and stop_sequence to see if there are any duplicate records for the same trip and stop sequence
print(f"Table of unique records for each combination of trip_id and stop_sequence in stop_times dataframe:")
display(stop_times.groupby(["trip_id", "stop_sequence"]).count())

for col in stop_times.columns:
    if col in ["trip_id", "stop_sequence"]:
        print(f"Distribution of {col} column in stop_times dataframe:")
        print(stop_times[col].value_counts(dropna=False))
        print("-" * 50)
    elif stop_times[col].dtype == "int64":
        print(f"Range of {col} column in stop_times dataframe: {stop_times[col].min()} to {stop_times[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in stop_times dataframe:")
        print(stop_times[col].value_counts(dropna=False))
        print("-" * 50)
    elif stop_times[col].dtype == "float64":
        print(f"Range of {col} column in stop_times dataframe: {stop_times[col].min()} to {stop_times[col].max()}")
        print("-" * 50)
    elif stop_times[col].dtype == "object":
        print(f"Distribution of {col} column in stop_times dataframe:")
        print(stop_times[col].value_counts(dropna=False))
        print("-" * 50)

Table of unique records for each combination of trip_id and stop_sequence in stop_times dataframe:


arrival_time  departure_time  stop_id  stop_headsign  \
trip_id   stop_sequence                                                         
1080      1                         1               1        1              1   
          2                         1               1        1              1   
          3                         1               1        1              1   
          4                         1               1        1              1   
          5                         1               1        1              1   
...                               ...             ...      ...            ...   
133251420 14                        1               1        1              0   
          15                        1               1        1              0   
          16                        1               1        1              0   
          17                        1               1        1              0   
          18                        1               1        1              0   

                         pickup_type  drop_off_type  shape_dist_traveled  \
trip_id   stop_sequence                                                    
1080      1                        1              1                    1   
          2                        1              1                    1   
          3                        1              1                    1   
          4                        1              1                    1   
          5                        1              1                    1   
...                              ...            ...                  ...   
133251420 14                       1              1                    1   
          15                       1              1                    1   
          16                       1              1                    1   
          17                       1              1                    1   
          18                       1              1                    1   

                         timepoint  
trip_id   stop_sequence             
1080      1                      1  
          2                      1  
          3                      1  
          4                      1  
          5                      1  
...                            ...  
133251420 14                     1  
          15                     1  
          16                     1  
          17                     1  
          18                     1  

[4283994 rows x 8 columns]

Distribution of trip_id column in stop_times dataframe:
trip_id
3179020      131
107399080    130
91239070     130
81494010     130
70041020     130
            ... 
18550010       1
60962010       1
93524020       1
84063010       1
29624020       1
Name: count, Length: 135408, dtype: int64
--------------------------------------------------
Distribution of arrival_time column in stop_times dataframe:
arrival_time
25:00:00    461
12:00:00    425
19:10:00    413
19:00:00    411
10:00:00    409
           ... 
28:29:44      1
29:11:49      1
29:10:06      1
29:06:30      1
28:55:41      1
Name: count, Length: 92155, dtype: int64
--------------------------------------------------
Distribution of departure_time column in stop_times dataframe:
departure_time
25:00:00    461
12:00:00    425
19:10:00    413
19:00:00    411
10:00:00    409
           ... 
28:29:44      1
29:11:49      1
29:10:06      1
29:06:30      1
28:55:41      1
Name: count, Length: 92155, dtype: int64
-------------------

In [828]:
# There are some obervations we can make from this

# 1) There are 4283994 rows in the stop_times dataframe and also 4283994 records with a unique combination of trip_id and stop_sequence, so the
#    are no duplicate records for the primary key of the stop_times dataframe

# 2) There are 135412 unique trip_ids in the trips dataframe but there are 135408 unique trip_ids in the stop_times dataframe, so there are
#    4 trip_ids in the trips dataframe that do not have any corresponding stop_times records. These trip_ids are seen below.
#    This may be due to the fact that some trips are scheduled but not currently active or in service, so they do not have any stop times 
#    associated with them yet; or the feed may just be missing stop_times records for these trip_ids.

# 3) There are 9423 unique stop_ids in the stop dataframe but only 9395 unique stop_ids in the stop_times dataframe, so there are 28
#    stop_ids missing in the stop_times dataframe. These stop_ids are seen below. These stops are quite possibly stops on the missing trips
#    excluded from the stop_times dataframe.

# 4) it seems that quite a few stop_headsigns are null for some reason (200k records out of 4.2 million), so to identify the route the trip is
#     taking, we would need to identify the trip_headsign of the corresponding trip_id in the trips dataframe

# 5) Most stops at most times offer pickup and dropoff services to passengers (value 0), but some do not (value 1), this is most likely when
#    the vehicle is performing a deadhead trip (not picking up or dropping off passengers between two scheduled points) or not in service

In [829]:
# find missing trip_ids in stop_times dataframe that are present in trips dataframe
missing_trip_ids = set(trips["trip_id"]) - set(stop_times["trip_id"])
print("Trip_ids present in trips dataframe but missing in stop_times dataframe:")
print(missing_trip_ids)
print("-" * 50)

# find missing stop_ids in stop_times dataframe that are present in stops dataframe
missing_stop_ids = set(stops["stop_id"]) - set(stop_times["stop_id"])
print("Stop_ids present in stops dataframe but missing in stop_times dataframe:")
print(missing_stop_ids)
print("-" * 50)

Trip_ids present in trips dataframe but missing in stop_times dataframe:
{85287020, 77572020, 80262020, 74591020}
--------------------------------------------------
Stop_ids present in stops dataframe but missing in stop_times dataframe:
{42506, 11155, 4635, 43036, 3742, 5429, 3772, 42819, 42825, 18379, 42956, 42957, 42958, 18384, 39379, 39380, 4822, 4823, 4825, 4826, 18396, 18397, 10077, 18399, 18400, 18797, 18798, 7295}
--------------------------------------------------


In [830]:
# lets look at the presence of null values and empty strings in the rows of the stop_times dataframe

print("Number of null values in each column of stop_times dataframe:")
print(stop_times.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of stop_times dataframe:")
object_columns = stop_times.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (stop_times[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

Number of null values in each column of stop_times dataframe:
trip_id                     0
arrival_time                0
departure_time              0
stop_id                     0
stop_sequence               0
stop_headsign          205764
pickup_type                 0
drop_off_type               0
shape_dist_traveled      8291
timepoint                   0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of stop_times dataframe:
arrival_time: 0
departure_time: 0
stop_headsign: 0
--------------------------------------------------


In [831]:
# we know that stop_headsign will have null values but why does shape_dist_traveled have null values?

# lets look at the frequency of unique values in the shape_dist_traveled column
print("Distribution of shape_dist_traveled column in stop_times dataframe:")
print(stop_times["shape_dist_traveled"].value_counts(dropna=False))
print("-" * 50)

# my initial hypothesis is that shape_dist_traveled is null for records first in the stop sequence of a trip. This is because the frequency of trips
# to have an initial distance traveled of 0 should be quite high and then the frequency distribution for indiivudal cumulative distances should be
# much lower since there are many different distances that can be traveled for different trips and stop sequences,
# since there is such a high frequency of null values, it is possible they are just a placeholder for  the distrance traveled value for a first stop

# lets look at some records with null shape_dist_traveled values
print("Records with null shape_dist_traveled values in stop_times dataframe:")
display(stop_times[stop_times["shape_dist_traveled"].isnull()].head(-10))
print("-" * 50)

# lets look at some records with trip_id = 133243013
print("Records with trip_id 133243013 in stop_times dataframe:")
display(stop_times[stop_times["trip_id"] == 133243013].head(10))
print("-" * 50)

# lets look at some records with trip_id = 133250505
print("Records with trip_id 133250505 in stop_times dataframe:")
display(stop_times[stop_times["trip_id"] == 133250505].head(10))
print("-" * 50)

# so from the 10 records shown, all contain a stop_sequence value of 1, so most likely they are the first stop in the stop sequence, and for 
# two of those trips, the first stop in the sequence has a null shape_dist_traveled value.

# lets see if all records with null shape_dist_traveled values have a stop_sequence value of 1
print("Distribution of stop_sequence values for records with null shape_dist_traveled values in stop_times dataframe:")
print(stop_times[stop_times["shape_dist_traveled"].isnull()]["stop_sequence"].value_counts())
print("-" * 50)

# so all records with null shape_dist_traveled values have a stop_sequence value of 1, so it seems that the initial hypothesis was correct 
# so lets impute the null values in the shape_dist_traveled column to 0...
stop_times["shape_dist_traveled"] = stop_times["shape_dist_traveled"].fillna(0)
print("Updated distribution of shape_dist_traveled column in stop_times dataframe:")
print(stop_times["shape_dist_traveled"].value_counts(dropna=False))
print("-" * 50)



Distribution of shape_dist_traveled column in stop_times dataframe:
shape_dist_traveled
0.0000       119525
NaN            8291
38.9837        2102
26.1379        2095
5.3470         1752
              ...  
2499.5300         1
2665.0900         1
2807.7900         1
3042.5500         1
1287.7800         1
Name: count, Length: 36321, dtype: int64
--------------------------------------------------
Records with null shape_dist_traveled values in stop_times dataframe:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
4078230,133243013,10:30:00,10:30:00,15547,1,NaN,0,0,NaN,1
4078280,133243012,09:00:00,09:00:00,15547,1,NaN,0,0,NaN,1
4078330,133243014,12:00:00,12:00:00,15547,1,NaN,0,0,NaN,1
4078351,133243015,13:14:27,13:14:27,11637,1,NaN,0,0,NaN,1
4078381,133243017,15:30:00,15:30:00,15547,1,NaN,0,0,NaN,1
...,...,...,...,...,...,...,...,...,...,...
4283529,133250503,24:27:44,24:27:44,13865,1,NaN,0,0,NaN,1
4283560,133250504,24:32:36,24:32:36,13865,1,NaN,0,0,NaN,1
4283591,133250505,24:37:28,24:37:28,13865,1,NaN,0,0,NaN,1
4283622,133250506,24:42:20,24:42:20,13865,1,NaN,0,0,NaN,1


--------------------------------------------------
Records with trip_id 133243013 in stop_times dataframe:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
4078230,133243013,10:30:00,10:30:00,15547,1,NaN,0,0,NaN,1
4078231,133243013,10:35:39,10:35:39,15601,2,NaN,0,0,1.7978,1
4078232,133243013,10:37:04,10:37:04,15589,3,NaN,0,0,2.0984,1
4078233,133243013,10:39:12,10:39:12,7980,4,NaN,0,0,2.5513,1
4078234,133243013,10:41:11,10:41:11,8522,5,NaN,0,0,2.9718,1
4078235,133243013,10:42:49,10:42:49,8811,6,NaN,0,0,3.3162,1
4078236,133243013,10:44:18,10:44:18,15602,7,NaN,0,0,3.6315,1
4078237,133243013,10:48:41,10:48:41,265,8,NaN,0,0,4.6768,1
4078238,133243013,10:50:42,10:50:42,7587,9,NaN,0,0,5.1729,1
4078239,133243013,10:51:39,10:51:39,7001,10,NaN,0,0,5.4098,1


--------------------------------------------------
Records with trip_id 133250505 in stop_times dataframe:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
4283591,133250505,24:37:28,24:37:28,13865,1,NaN,0,0,NaN,1
4283592,133250505,24:41:05,24:41:05,13731,2,NaN,0,0,2.7446,1
4283593,133250505,24:43:57,24:43:57,13733,3,NaN,0,0,4.9228,1
4283594,133250505,24:45:34,24:45:34,13736,4,NaN,0,0,6.1462,1
4283595,133250505,24:46:53,24:46:53,13737,5,NaN,0,0,7.1231,1
4283596,133250505,24:47:53,24:47:53,13740,6,NaN,0,0,7.9361,1
4283597,133250505,24:49:13,24:49:13,13741,7,NaN,0,0,8.4659,1
4283598,133250505,24:50:13,24:50:13,13744,8,NaN,0,0,9.2348,1
4283599,133250505,24:51:32,24:51:32,13745,9,NaN,0,0,9.7841,1
4283600,133250505,24:52:53,24:52:53,13747,10,NaN,0,0,10.3460,1


--------------------------------------------------
Distribution of stop_sequence values for records with null shape_dist_traveled values in stop_times dataframe:
stop_sequence
1    8291
Name: count, dtype: int64
--------------------------------------------------
Updated distribution of shape_dist_traveled column in stop_times dataframe:
shape_dist_traveled
0.0000       127816
38.9837        2102
26.1379        2095
5.3470         1752
10.6211        1095
              ...  
2499.5300         1
2665.0900         1
2807.7900         1
3042.5500         1
1144.9700         1
Name: count, Length: 36320, dtype: int64
--------------------------------------------------


In [832]:
# lets re-look at the presence of null values and empty strings in the rows of the stop_times dataframe

print("Number of null values in each column of stop_times dataframe:")
print(stop_times.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of stop_times dataframe:")
object_columns = stop_times.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (stop_times[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

Number of null values in each column of stop_times dataframe:
trip_id                     0
arrival_time                0
departure_time              0
stop_id                     0
stop_sequence               0
stop_headsign          205764
pickup_type                 0
drop_off_type               0
shape_dist_traveled         0
timepoint                   0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of stop_times dataframe:
arrival_time: 0
departure_time: 0
stop_headsign: 0
--------------------------------------------------


#### Shapes.txt

In [833]:
# print data types of shapes dataframe

print("Data types of shapes dataframe:")
print(shapes.dtypes)
print("-" * 50)

Data types of shapes dataframe:
shape_id                object
shape_pt_lat           float64
shape_pt_lon           float64
shape_pt_sequence        int64
shape_dist_traveled    float64
dtype: object
--------------------------------------------------


In [834]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in shapes.columns:
    unique_values = shapes[col].unique()
    print(f"Unique values in {col} column of shapes dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in shape_id column of shapes dataframe:
['shp-100-01' 'shp-100-02' 'shp-100-03' ... '1103678' '1103684' '1103685']
--------------------------------------------------
Unique values in shape_pt_lat column of shapes dataframe:
[43.701911 43.70207  43.702164 ... 43.762777 43.763014 43.76327 ]
--------------------------------------------------
Unique values in shape_pt_lon column of shapes dataframe:
[-79.35283  -79.352713 -79.352644 ... -79.492964 -79.492263 -79.49189 ]
--------------------------------------------------
Unique values in shape_pt_sequence column of shapes dataframe:
[   1    2    3 ... 3068 3069 3070]
--------------------------------------------------
Unique values in shape_dist_traveled column of shapes dataframe:
[ 0.     20.     31.86   ...  7.0991  7.1741  7.5274]
--------------------------------------------------


In [835]:
# before we realized that shape_id had duplicated string and integer values for the same shape_id for Trips, 
# lets check if that is still the case here

print("Unique values in shape_id column of shape dataframe:")
print(shapes["shape_id"].unique().tolist())

# it looks like all shape_id's are strings, but lets just make sure by checking if we get the same id as a string and integer

display(shapes[(shapes["shape_id"] == 1103629)].head(3))
display(shapes[(shapes["shape_id"] == "1103629")].head(3))

# looks like we dont need to worry about redundant records in the shape dataframe due to the same shape_id being represented as both a string and
# integer


Unique values in shape_id column of shape dataframe:
['shp-100-01', 'shp-100-02', 'shp-100-03', 'shp-100-04', 'shp-10-01', 'shp-10-02', 'shp-10-03', 'shp-10-04', 'shp-100-51', 'shp-100-52', 'shp-100-53', 'shp-100-54', 'shp-1-01', 'shp-101-01', 'shp-101-02', 'shp-101-03', 'shp-101-05', 'shp-101-06', 'shp-101-07', 'shp-101-51', 'shp-101-52', 'shp-101-53', 'shp-101-54', 'shp-101-56', 'shp-101-57', 'shp-101-58', 'shp-102-01', 'shp-102-02', 'shp-102-03', 'shp-102-05', 'shp-102-06', 'shp-102-08', 'shp-102-11', 'shp-102-51', 'shp-102-52', 'shp-102-54', 'shp-102-55', 'shp-102-56', 'shp-102-57', 'shp-102-59', 'shp-103-01', 'shp-103-51', 'shp-103-52', 'shp-104-02', 'shp-104-52', 'shp-105-07', 'shp-10-51', 'shp-10-52', 'shp-105-55', 'shp-106-01', 'shp-106-03', 'shp-106-54', 'shp-106-57', 'shp-107-07', 'shp-107-33', 'shp-107-53', 'shp-107-73', 'shp-108-04', 'shp-108-05', 'shp-108-06', 'shp-108-52', 'shp-108-53', 'shp-108-54', 'shp-108-55', 'shp-108-56', 'shp-109-06', 'shp-109-07', 'shp-109-08', 's

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
1018493,1103629,43.769143,-79.376846,1,0.0000
1018494,1103629,43.769421,-79.375510,2,0.1124
1018495,1103629,43.769658,-79.374399,3,0.2054


In [836]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

# groupby shape_id and shape_pt_sequence to see if there are any duplicate records for the same trip and stop sequence
print(f"Table of unique records for each combination of shape_id and shape_pt_sequence in shapes dataframe:")
display(shapes.groupby(["shape_id", "shape_pt_sequence"]).count())

for col in shapes.columns:
    if col in ["shape_id", "shape_pt_sequence"]:
        print(f"Distribution of {col} column in shapes dataframe:")
        print(shapes[col].value_counts(dropna=False))
        print("-" * 50)
    elif shapes[col].dtype == "int64":
        print(f"Range of {col} column in shapes dataframe: {shapes[col].min()} to {shapes[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in shapes dataframe:")
        print(shapes[col].value_counts(dropna=False))
        print("-" * 50)
    elif shapes[col].dtype == "float64":
        print(f"Range of {col} column in shapes dataframe: {shapes[col].min()} to {shapes[col].max()}")
        print("-" * 50)
    elif shapes[col].dtype == "object":
        print(f"Distribution of {col} column in shapes dataframe:")
        print(shapes[col].value_counts(dropna=False))
        print("-" * 50)


Table of unique records for each combination of shape_id and shape_pt_sequence in shapes dataframe:


shape_pt_lat  shape_pt_lon  shape_dist_traveled
shape_id   shape_pt_sequence                                                 
1101825    1                             1             1                    1
           2                             1             1                    1
           3                             1             1                    1
           4                             1             1                    1
           5                             1             1                    1
...                                    ...           ...                  ...
shp-996-53 1055                          1             1                    1
           1056                          1             1                    1
           1057                          1             1                    1
           1058                          1             1                    1
           1059                          1             1                    1

[1020671 rows x 3 columns]

Distribution of shape_id column in shapes dataframe:
shape_id
shp-334-61    3070
shp-334-58    3036
shp-334-52    3015
shp-334-64    2882
shp-300-97    2805
              ... 
shp-63-24       32
1103629         30
shp-79-56       26
shp-55-02       23
1103630         22
Name: count, Length: 1565, dtype: int64
--------------------------------------------------
Range of shape_pt_lat column in shapes dataframe: 43.591495 to 43.90975
--------------------------------------------------
Range of shape_pt_lon column in shapes dataframe: -79.650423 to -79.122627
--------------------------------------------------
Distribution of shape_pt_sequence column in shapes dataframe:
shape_pt_sequence
21      1563
22      1563
23      1562
20      1562
19      1561
        ... 
3047       1
3048       1
3050       1
3051       1
3070       1
Name: count, Length: 3070, dtype: int64
--------------------------------------------------
Range of shape_dist_traveled column in shapes dataframe: 0.0 to 50677.14
--

In [837]:
# There are some obervations we can make from this

# 1) There are 1020671 rows in the shapes dataframe and also 1020671 records with a unique combination of shape_id and shape_pt_sequence, so there
#    are no duplicate records for the primary key of the shapes dataframe

# 2) as mentioned before it is odd that the shape_id column is an object column but that is just because some of the id's are annotated as 
#    shp-###-##

# 3) The range of shape_pt_lat and shape_pt_lon seem to align with toronto's coordinates, so thats a good sign that the feed information is 
#    complete and accurate. The ranges listed in shapes dataframe is slightly different than those listed on stops (43.591495 to 43.90975 in shapes
#    vs 43.59181 to 43.90975 in stops and -79.650423 to -79.122627 in shapes vs -79.649908 to -79.123111 in stops), but this is fine since in the
#    GTFS documentation, coordinates of stops and shapes may be slightly different 


# 4) It is a bit strange the the most frequent values for shape_pt_sequence is in the 20's and not just 1 (you would think the first points in the
#   sequence would have low number for shape_pt_sequence) (see below)

# 5) The range of the shape_dist_traveled column in the shapes dataframe matches the range of the shape_dist_traveled in the Stop_times dataframe
#    which is a good indicator of data consistency in the feed (0.0 to 50677.14)


In [838]:
# view full distribution of shape_pt_sequence column
print(shapes["shape_pt_sequence"].value_counts(dropna=False).sort_index().to_string())

# so there are 1554 records with a shape_pt_sequence of 1 but 1563 records with shape_pt_sequence of 22
# lets view records whose shape_id has a shape_pt_sequence of 22 but not 1

valid_shape_ids = (
    shapes.groupby("shape_id")["shape_pt_sequence"]
    .apply(lambda x: 22 in x.values and 1 not in x.values)
)

valid_ids = valid_shape_ids[valid_shape_ids].index

print("shape_ids with a shape_pt_sequence of 22 but not 1")
display(shapes[shapes["shape_id"].isin(valid_ids)])

# so it looks like a few unique shapes have a shape_pt_sequence start with not 1 
print("shape_id of shp-334-51")
display(shapes[shapes["shape_id"] == "shp-334-51"])

# for example "shp-334-51" starts with a shape_pt_sequence of 3, so it is technically plausible that non "1" numbers have a higher record-frequency
# count than 1 (shape_pt_sequence values of 1)

shape_pt_sequence
1       1554
2       1556
3       1557
4       1557
5       1559
6       1560
7       1560
8       1560
9       1560
10      1560
11      1560
12      1561
13      1561
14      1561
15      1561
16      1561
17      1561
18      1561
19      1561
20      1562
21      1563
22      1563
23      1562
24      1561
25      1561
26      1561
27      1560
28      1560
29      1560
30      1560
31      1559
32      1559
33      1559
34      1559
35      1559
36      1558
37      1558
38      1558
39      1557
40      1557
41      1555
42      1555
43      1554
44      1554
45      1554
46      1554
47      1554
48      1553
49      1553
50      1553
51      1552
52      1552
53      1551
54      1549
55      1545
56      1543
57      1539
58      1539
59      1538
60      1537
61      1537
62      1536
63      1534
64      1534
65      1532
66      1529
67      1527
68      1526
69      1523
70      1521
71      1518
72      1517
73      1516
74      1515
75      1514
76     

,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
367032,shp-334-51,43.681991,-79.610686,3,40.00
367033,shp-334-51,43.682027,-79.610636,4,45.68
367034,shp-334-51,43.682056,-79.610583,5,51.02
367035,shp-334-51,43.682118,-79.610468,6,62.55
367036,shp-334-51,43.682171,-79.610386,7,71.39
...,...,...,...,...,...
858336,shp-952-04,43.687240,-79.621414,101,1531.69
858337,shp-952-04,43.687070,-79.621329,102,1551.69
858338,shp-952-04,43.687022,-79.621305,103,1557.42
858339,shp-952-04,43.686911,-79.621109,104,1577.42


shape_id of shp-334-51


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
367032,shp-334-51,43.681991,-79.610686,3,40.00
367033,shp-334-51,43.682027,-79.610636,4,45.68
367034,shp-334-51,43.682056,-79.610583,5,51.02
367035,shp-334-51,43.682118,-79.610468,6,62.55
367036,shp-334-51,43.682171,-79.610386,7,71.39
...,...,...,...,...,...
367123,shp-334-51,43.687364,-79.621110,94,1457.65
367124,shp-334-51,43.687218,-79.621060,95,1474.37
367125,shp-334-51,43.687043,-79.621005,96,1494.37
367126,shp-334-51,43.686867,-79.620950,97,1514.37


In [839]:
# lets look at the presence of null values and empty strings in the rows of the shapes dataframe

print("Number of null values in each column of shapes dataframe:")
print(shapes.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of shapes dataframe:")
object_columns = shapes.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (shapes[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# looks like the feed is complete


Number of null values in each column of shapes dataframe:
shape_id               0
shape_pt_lat           0
shape_pt_lon           0
shape_pt_sequence      0
shape_dist_traveled    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of shapes dataframe:
shape_id: 0
--------------------------------------------------


#### Calendar.txt

In [ ]:
print("Data types of calendar dataframe:")
print(calendar.dtypes)
print("-" * 50)

print("Data types of calendar_dates dataframe:")
print(calendar_dates.dtypes)
print("-" * 50)

print("Data types of route_types dataframe:")
print(route_types.dtypes)
print("-" * 50)

Data types of calendar dataframe:
service_id    int64
monday        int64
tuesday       int64
wednesday     int64
thursday      int64
friday        int64
saturday      int64
sunday        int64
start_date    int64
end_date      int64
dtype: object
--------------------------------------------------
Data types of agency dataframe:
agency_id           int64
agency_name        object
agency_url         object
agency_timezone    object
agency_lang        object
agency_phone       object
agency_fare_url    object
agency_email       object
cemv_support        int64
dtype: object
--------------------------------------------------
Data types of calendar_dates dataframe:
service_id        int64
date              int64
exception_type    int64
dtype: object
--------------------------------------------------
Data types of route_types dataframe:
Code              object
Transport_Mode    object
Service_Type      object
Sch_Avail         object
DirectionName     object
dtype: object
-----------------